<a href="https://colab.research.google.com/github/tommbk50-hub/Cheminformatics/blob/main/RetroDFM_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RetroDFM-R: Reasoning-Driven Retrosynthesis Prediction with Large Language Models via Reinforcement Learning


* Retrosynthetic planning is a cornerstone of organic synthesis and drug discovery. Yet existing AI methods often rely on pattern matching rather than transferable chemical reasoning, limiting both generalizability and interpretability.

* Here we introduce RetroDFM-R, a
reasoning-driven large language model (LLM) for chemical retrosynthesis. Leveraging largescale reinforcement learning, RetroDFM-R moves beyond black-box prediction by coupling
improved accuracy with transparent, step-by-step rationale. On the USPTO-50K benchmark,
RetroDFM-R achieves 60.4% accuracy without augmentation and 66.1% with the full inference
setup, outperforming previous state-of-the-art baselines.

* Beyond standard metrics, double-blind
expert evaluation further supports the chemical plausibility and practical utility of its proposed
pathways. We also demonstrate that RetroDFM-R can reconstruct complex, multistep synthetic routes for real-world pharmaceuticals and self-assembled monolayer materials.

* By making
its reasoning explicit and human-interpretable, RetroDFM-R addresses a key barrier to trust
and supports practical deployment in automated retrosynthetic planning.

**Switch to an L4 GPU Instance before starting**

In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

In [ ]:
import torch

assert torch.cuda.is_available(), "Still on CPU! Go to Runtime > Change runtime type > L4 GPU."
print(f"Connected to: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")

# Cell 1: Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes rdkit

# Cell 2: Load the model and tokenizer
Set USE_4BIT = True if you are on a T4 GPU; leave it False for L4 or A100 runtimes.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "OpenDFM/RetroDFM-R-8B"
USE_4BIT = False  # Set to True for T4 (16GB VRAM)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quant_config,
        device_map="auto",
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

# Cell 3: Run single-step retrosynthesis inference
This uses the example product SMILES from Figure 1(c) of the paper.

In [ ]:
import re
from transformers import TextStreamer

def predict_retrosynthesis(product_smiles, max_new_tokens=2048, temperature=0.6):
    prompt = (
        f"<SMILES>{product_smiles}</SMILES>\n"
        "Given the product SMILES, your task is to predict the reactants SMILES "
        "using your experienced chemical retrosynthesis knowledge. "
        "Please reason step by step, and put your final answer within <answer> answer here </answer>."
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Prints tokens live to the cell output as they are generated
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            streamer=streamer,
        )

    return tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )

product_smiles = "CCOC(=O)c1sc(N2CCOCC2)nc1CBr"
output = predict_retrosynthesis(product_smiles)

# Cell 4: Parse and visualize the predicted reactants with RDKit

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

match = re.search(r"<answer>\s*(.*?)\s*</answer>", output, re.DOTALL)
if match:
    reactants_smiles = match.group(1).strip()
    print("Predicted Reactants:", reactants_smiles)

    mols = [Chem.MolFromSmiles(product_smiles)]
    legends = ["Target Product"]

    for i, smi in enumerate(reactants_smiles.split("."), 1):
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mols.append(mol)
            legends.append(f"Reactant {i}")

    img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 300), legends=legends)
    display(img)
else:
    print("Could not find <answer> tags in the model output.")

In [ ]:
import re
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw, rdMolDescriptors
from IPython.display import display, Markdown

# --- Configurable ELN Scale Parameters ---
TARGET_PRODUCT_G = 1.0     # Desired product mass (g)
ASSUMED_YIELD = 0.80       # Used to back-calculate starting material scale
REAGENT_EQUIV = 1.10       # Default equivalents for secondary reagents (e.g., NBS)

# --- Modular RDKit Tool Functions (MCP-ready) ---
def rdkit_parse_retro_output(text):
    think_m = re.search(r"<think>\s*(.*?)\s*</think>", text, re.DOTALL)
    ans_m = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.DOTALL)
    rationale = think_m.group(1).strip() if think_m else "No rationale found."
    reactants = ans_m.group(1).strip() if ans_m else None
    return rationale, reactants

def rdkit_reaction_scheme(reactants_smi, product_smi):
    rxn = AllChem.ReactionFromSmarts(f"{reactants_smi}>>{product_smi}", useSmiles=True)
    return Draw.ReactionToImage(rxn, subImgSize=(320, 220))

def rdkit_stoichiometry_table(reactants_smi, product_smi, target_g=1.0, est_yield=0.80, reagent_eq=1.10):
    prod_mol = Chem.MolFromSmiles(product_smi)
    prod_mw = Descriptors.MolWt(prod_mol)
    target_mmol = (target_g / prod_mw) * 1000.0
    limiting_mmol = target_mmol / est_yield

    reactant_list = [s.strip() for s in reactants_smi.split(".") if s.strip()]
    parsed = [(s, Chem.MolFromSmiles(s)) for s in reactant_list]
    parsed = [(s, m) for s, m in parsed if m is not None]

    # Identify limiting substrate as the largest fragment (MaxFrag)
    limiting_idx = max(range(len(parsed)), key=lambda i: parsed[i][1].GetNumHeavyAtoms())

    rows = []
    for i, (smi, mol) in enumerate(parsed):
        mw = Descriptors.MolWt(mol)
        formula = rdMolDescriptors.CalcMolFormula(mol)
        is_lim = (i == limiting_idx)
        equiv = 1.00 if is_lim else reagent_eq
        mmol = limiting_mmol * equiv
        mass_g = (mmol * mw) / 1000.0
        rows.append({
            "Role": "Limiting Substrate" if is_lim else f"Reagent {i}",
            "Formula": formula,
            "MW (g/mol)": round(mw, 2),
            "Equiv": equiv,
            "mmol": round(mmol, 2),
            "Mass (g)": round(mass_g, 3),
            "SMILES": smi,
        })

    rows.append({
        "Role": f"Target Product ({int(est_yield*100)}% est.)",
        "Formula": rdMolDescriptors.CalcMolFormula(prod_mol),
        "MW (g/mol)": round(prod_mw, 2),
        "Equiv": est_yield,
        "mmol": round(target_mmol, 2),
        "Mass (g)": round(target_g, 3),
        "SMILES": product_smi,
    })
    return pd.DataFrame(rows)

def rdkit_safety_and_selectivity(reactants_smi, product_smi):
    rules = [
        ("N-Haloimide (e.g., NBS/NCS)", "[#7]([Br,Cl,I])C(=O)",
         "Safety: Exothermic radical initiation; moisture/light sensitive. Keep away from strong bases/reducing agents.",
         "Selectivity: Risk of over-bromination (dibromination) or competing electrophilic aromatic bromination on electron-rich rings."),
        ("Alkyl / Benzylic Halide", "[CX4][Br,I]",
         "Safety: Potential lachrymator and alkylating agent (mutagenicity alert). Handle strictly in a fume hood.",
         "Selectivity: Product halide is susceptible to hydrolysis or nucleophilic displacement during aqueous workup."),
        ("Ester Functionality", "[CX3](=O)[OX2][#6]",
         "Safety: Generally low acute hazard.",
         "Selectivity: Avoid strong aqueous acid/base during workup to prevent ester hydrolysis or transesterification."),
        ("Tertiary / Heteroaryl Amine", "[NX3;H0;!$(NC=O)]",
         "Safety: Potential irritant.",
         "Selectivity: Electron-rich amine/morpholine sites can undergo oxidative degradation or N-halogenation with excess oxidant/NBS."),
        ("Nitro Group", "[$([NX3](=O)=O),$([NX3+](=O)[O-])]",
         "Safety: Energetic functional group; assess thermal runaway risk on scale.",
         "Selectivity: Sensitive to reductive conditions."),
        ("Organic Azide", "[$([NX2]=[NX2+]=[NX1-]),$([NX1-]=[NX2+]=[NX2])]",
         "Safety: Shock/heat-sensitive explosive hazard; avoid concentration to dryness and halogenated solvents (DCM).",
         "Selectivity: Reacts readily with phosphines (Staudinger) and alkynes."),
    ]

    all_mols = [("Reactants", Chem.MolFromSmiles(reactants_smi)), ("Product", Chem.MolFromSmiles(product_smi))]
    safety_notes, selectivity_notes = [], []

    for name, smarts, safe_msg, sel_msg in rules:
        pat = Chem.MolFromSmarts(smarts)
        hits = [label for label, m in all_mols if m and m.HasSubstructMatch(pat)]
        if hits:
            loc = "/".join(hits)
            safety_notes.append(f"- **{name} ({loc}):** {safe_msg}")
            selectivity_notes.append(f"- **{name} ({loc}):** {sel_msg}")

    if not safety_notes:
        safety_notes.append("- No high-priority SMARTS hazard alerts triggered; follow standard organic lab PPE and fume hood protocols.")
    if not selectivity_notes:
        selectivity_notes.append("- No major functional-group incompatibility alerts detected via SMARTS screen.")
    return safety_notes, selectivity_notes

# --- Generate & Render Full ELN Entry ---
rationale, reactants_smiles = rdkit_parse_retro_output(output)

if reactants_smiles:
    display(Markdown("## Electronic Lab Notebook (ELN) — Retrosynthesis Write-Up"))
    display(Markdown("### 1. Reaction Scheme"))
    display(rdkit_reaction_scheme(reactants_smiles, product_smiles))

    stoich_df = rdkit_stoichiometry_table(
        reactants_smiles, product_smiles,
        target_g=TARGET_PRODUCT_G, est_yield=ASSUMED_YIELD, reagent_eq=REAGENT_EQUIV
    )
    safety_notes, selectivity_notes = rdkit_safety_and_selectivity(reactants_smiles, product_smiles)

    eln_md = f"""
### 2. Stoichiometry Table
*(Scaled for **{TARGET_PRODUCT_G:.2f} g** target product assuming **{int(ASSUMED_YIELD*100)}%** isolated yield)*

{stoich_df.to_markdown(index=False)}

### 3. RetroDFM-R Mechanistic & Disconnection Rationale
{rationale}

### 4. Selectivity & Functional Group Compatibility
{chr(10).join(selectivity_notes)}

### 5. Safety & Hazard Warnings (RDKit SMARTS Screen)
{chr(10).join(safety_notes)}
"""
    display(Markdown(eln_md))
else:
    print("Could not find <answer> tags in the model output.")

In [ ]:
import re
import torch
from threading import Thread
from transformers import TextIteratorStreamer
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display, Markdown

# --- Selectivity Site SMARTS Rules: (Label, Keywords, SMARTS, RGB, Hex) ---
SELECTIVITY_RULES = [
    (
        "Target Benzylic/Heteroarylic Methyl (Primary Site)",
        ["methyl", "benzylic", "radical", "bromination", "nbs"],
        "[c][CH3]",
        (0.20, 0.78, 0.35), "#33C759",
    ),
    (
        "Bromomethyl / Over-Bromination Site (Dibromination Risk)",
        ["dibrom", "over-brom", "bromomethyl", "halide", "c-br"],
        "[c][CH2][Br,Cl,I]",
        (1.00, 0.58, 0.00), "#FF9400",
    ),
    (
        "Morpholine / Tertiary Amine (Oxidation / N-Halogenation Risk)",
        ["morpholine", "amine", "degradation", "oxidation", "nitrogen"],
        "C1COCCN1",
        (0.92, 0.22, 0.22), "#EB3838",
    ),
    (
        "Ester / Carbonyl Group (Hydrolysis / Alpha-Side Reaction)",
        ["ester", "carbonyl", "alpha", "hydrolysis"],
        "[CX3](=O)[OX2][#6]",
        (0.20, 0.52, 0.96), "#3385F5",
    ),
    (
        "Unsubstituted Aromatic CH (Competing EAS Site)",
        ["electrophilic", "aromatic", "ring", "eas"],
        "[cH]",
        (0.66, 0.32, 0.88), "#A852E0",
    ),
]

def render_selectivity_highlights(reactants_smi, product_smi, discussion_text):
    text_lower = discussion_text.lower()

    # Pick limiting substrate (largest fragment) and product
    frag_mols = [Chem.MolFromSmiles(s.strip()) for s in reactants_smi.split(".") if s.strip()]
    frag_mols = [m for m in frag_mols if m is not None]
    substrate_mol = max(frag_mols, key=lambda m: m.GetNumHeavyAtoms())
    product_mol = Chem.MolFromSmiles(product_smi)

    mols = [substrate_mol, product_mol]
    legends = ["Limiting Substrate", "Target Product"]
    hl_atoms_list, hl_colors_list = [], []
    active_legend = {}

    for mol in mols:
        atom_colors = {}
        for label, keywords, smarts, rgb, hex_col in SELECTIVITY_RULES:
            if any(kw in text_lower for kw in keywords):
                pat = Chem.MolFromSmarts(smarts)
                if pat:
                    for match in mol.GetSubstructMatches(pat):
                        for idx in match:
                            atom_colors[idx] = rgb
                        active_legend[label] = hex_col
        hl_atoms_list.append(list(atom_colors.keys()))
        hl_colors_list.append(atom_colors)

    if not active_legend:
        return

    legend_md = "### Highlighted Selectivity & Side-Reaction Sites\n" + "\n".join(
        f"- <span style='color:{hex_col}; font-weight:bold;'>&#9632;</span> **{label}**"
        for label, hex_col in active_legend.items()
    )
    display(Markdown(legend_md))

    img = Draw.MolsToGridImage(
        mols,
        molsPerRow=2,
        subImgSize=(380, 280),
        legends=legends,
        highlightAtomLists=hl_atoms_list,
        highlightAtomColors=hl_colors_list,
    )
    display(img)


class WetLabAssistant:
    def __init__(self, product_smi, reactants_smi, rationale):
        self.product_smi = product_smi
        self.reactants_smi = reactants_smi
        system_prompt = (
            "You are an expert synthetic organic chemist assisting with wet-lab troubleshooting. "
            "You are discussing a specific reaction from an Electronic Lab Notebook (ELN):\n"
            f"- Target Product SMILES: {product_smi}\n"
            f"- Planned Reactants SMILES: {reactants_smi}\n"
            f"- Original Rationale: {rationale}\n\n"
            "When the user reports actual experimental observations (e.g., low yield, TLC, LC-MS m/z, "
            "NMR peaks, side products, or solubility issues), respond in clean Markdown with concise "
            "sections for: **Mechanistic Diagnosis**, **Recommended Condition Adjustments** (equivalents, "
            "initiator, solvent, temperature, workup), and **Alternative Route / Disconnection**."
        )
        self.messages = [{"role": "system", "content": system_prompt}]

    @staticmethod
    def _format_markdown(raw_text, observation):
        think_m = re.search(r"<think>(.*?)(?:</think>|$)", raw_text, re.DOTALL)
        if think_m:
            think_text = think_m.group(1).strip()
            answer_text = raw_text[think_m.end():].strip()
            think_block = (
                "<details>\n<summary><b>Model Reasoning (&lt;think&gt;)</b></summary>\n\n"
                f"{think_text}\n</details>\n\n"
            )
        else:
            think_block = ""
            answer_text = raw_text.strip()

        return (
            "## Wet-Lab Troubleshooting Log\n"
            f"**Reported Observation:** {observation}\n\n"
            f"{think_block}"
            "### Chemist Feedback & Adjustments\n"
            f"{answer_text if answer_text else '*Thinking...*'}"
        )

    def report(self, lab_observation, max_new_tokens=1536, temperature=0.6):
        self.messages.append({"role": "user", "content": lab_observation})
        text = tokenizer.apply_chat_template(
            self.messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer([text], return_tensors="pt").to(model.device)
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

        gen_kwargs = dict(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            streamer=streamer,
        )

        thread = Thread(target=model.generate, kwargs=gen_kwargs)
        thread.start()

        handle = display(Markdown("*Analyzing wet-lab data...*"), display_id=True)
        reply = ""
        for chunk in streamer:
            reply += chunk
            handle.update(Markdown(self._format_markdown(reply, lab_observation)))

        thread.join()
        self.messages.append({"role": "assistant", "content": reply})

        # Render RDKit molecular highlights based on the selectivity discussion
        render_selectivity_highlights(
            self.reactants_smi, self.product_smi, f"{lab_observation}\n{reply}"
        )
        return reply

    def reset(self):
        self.messages = self.messages[:1]
        display(Markdown("*Conversation history cleared (ELN context retained).*"))


# Initialize with the reaction from Cells 3 & 4
lab_chat = WetLabAssistant(product_smiles, reactants_smiles, rationale)

# Example: feed back actual wet-lab observations
_ = lab_chat.report(
    "I ran this with 1.1 eq NBS and 0.1 eq AIBN in MeCN at 80 °C for 4 h. "
    "LC-MS shows ~35% desired product ([M+H]+ = 335/337), ~40% dibrominated "
    "byproduct ([M+H]+ = 413/415/417), and some baseline morpholine degradation. "
    "How should I modify the conditions or route to improve selectivity?"
)

In [ ]:
!pip install -q -U google-genai

import os
from google import genai
from google.genai import types
from google.colab import userdata
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display, Markdown

# --- Dynamic RDKit Tool (MCP-compatible, zero hardcoded rules) ---
PALETTE = [
    ((0.20, 0.78, 0.35), "#33C759"),  # Green
    ((1.00, 0.58, 0.00), "#FF9400"),  # Orange
    ((0.92, 0.22, 0.22), "#EB3838"),  # Red
    ((0.20, 0.52, 0.96), "#3385F5"),  # Blue
    ((0.66, 0.32, 0.88), "#A852E0"),  # Purple
]

def draw_selectivity_sites(
    smiles: str,
    title: str,
    site_labels: list[str],
    site_smarts: list[str],
) -> str:
    """Renders a 2D molecule with color-coded atom highlights for selectivity analysis.

    Args:
        smiles: Valid SMILES string of the molecule to draw (substrate, product, or byproduct).
        title: Short title for the figure (e.g., 'Limiting Substrate' or 'Dibrominated Byproduct').
        site_labels: List of human-readable descriptions for each reactive/selectivity site.
        site_smarts: List of RDKit SMARTS patterns corresponding 1-to-1 with site_labels.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return f"Error: Invalid SMILES '{smiles}'. Please fix the SMILES and retry."

    if len(site_labels) != len(site_smarts):
        return "Error: site_labels and site_smarts must have the same length."

    atom_colors = {}
    matched_legend = []
    warnings = []

    for i, (label, smarts) in enumerate(zip(site_labels, site_smarts)):
        pat = Chem.MolFromSmarts(smarts)
        if pat is None:
            warnings.append(f"Invalid SMARTS '{smarts}' for '{label}'")
            continue
        matches = mol.GetSubstructMatches(pat)
        if not matches:
            warnings.append(f"SMARTS '{smarts}' matched 0 atoms on {smiles}")
            continue
        rgb, hex_col = PALETTE[i % len(PALETTE)]
        for match in matches:
            for idx in match:
                atom_colors[idx] = rgb
        matched_legend.append((label, hex_col, len(matches)))

    if not matched_legend:
        return "No SMARTS matched. Issues: " + "; ".join(warnings) + ". Adjust SMARTS and retry."

    legend_md = f"#### {title}\n" + "\n".join(
        f"- <span style='color:{hex_col}; font-weight:bold;'>&#9632;</span> **{label}**"
        for label, hex_col, _ in matched_legend
    )
    display(Markdown(legend_md))

    img = Draw.MolToImage(
        mol,
        size=(420, 300),
        highlightAtoms=list(atom_colors.keys()),
        highlightAtomColors=atom_colors,
    )
    display(img)

    status = f"Rendered '{title}' with {len(matched_legend)} highlighted site(s)."
    if warnings:
        status += " Warnings: " + "; ".join(warnings)
    return status


# --- Initialize Gemini Client & Stateful Wet-Lab Chat ---
api_key = userdata.get("KDD_GEMINI_API_KEY") or os.environ.get("KDD_GEMINI_API_KEY")
# Explicitly set the API version to v1beta to support function calling
client = genai.Client(api_key=api_key, http_options={"api_version": "v1beta"})

system_instruction = (
    "You are an expert synthetic organic chemist assisting with wet-lab troubleshooting "
    "for an Electronic Lab Notebook (ELN).\n"
    f"- Target Product SMILES: {product_smiles}\n"
    f"- Planned Reactants SMILES: {reactants_smiles}\n"
    f"- RetroDFM-R Rationale: {rationale}\n\n"
    "When the user reports experimental data (TLC, LC-MS, NMR, side products, low yield):\n"
    "1. Call `draw_selectivity_sites` to visually highlight the exact reactive sites, "
    "competing side-reaction sites, or proposed byproduct structures using valid RDKit SMARTS.\n"
    "2. Provide a clean Markdown response with sections for **Mechanistic Diagnosis**, "
    "**Recommended Condition Adjustments**, and **Alternative Route / Disconnection**."
)

lab_chat = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        tools=[draw_selectivity_sites],
        temperature=0.4,
    ),
)

def report_lab_data(observation: str):
    display(Markdown(f"## Wet-Lab Troubleshooting Log\n**Reported Observation:** {observation}"))
    response = lab_chat.send_message(observation)
    display(Markdown(response.text))

# Example: feed back actual wet-lab observations
report_lab_data(
    "I ran this with 1.1 eq NBS and 0.1 eq AIBN in MeCN at 80 C for 4 h. "
    "LC-MS shows ~35% desired product ([M+H]+ = 335/337), ~40% dibrominated "
    "byproduct ([M+H]+ = 413/415/417), and some baseline morpholine degradation. "
    "How should I modify the conditions or route to improve selectivity?"
)